# Matched baseline versus AuxK on archived MRI embeddings

No encoder retraining. Reuse the SAME verified `rank_control_embeddings_v1.zip`
already uploaded to `YOUR_USERNAME/YOUR_PRIVATE_DATASET`; no new dataset upload is needed.
Add your read-only HF_TOKEN to runtime secrets or .env as before.

Native training-mean-centered 192-dimensional CLS only. NCL and SimCLR, encoder
seeds 10/42/52, SAE seeds 10/42/52, baseline and AuxK: **36 fresh SAE fits**.
One encoder seed gives **12 fits**. Default encoder seed is 10; then run 42 and 52.
The old rank-control results stay in their original folder. Use the fresh default
`auxk_native_controls_v1` output folder for this experiment.

Both arms share initialization, shuffle order, D=1536, K=24, 30 epochs,
batch=512, Adam LR=0.0005 and gradient clipping=1. AuxK alone adds residual
reconstruction using positive Top-K activations among training-inactive latents.
The residual target is detached; the auxiliary decoder has no output bias.
Alpha=1/32, K_aux=192, inactivity window=one full training pass (138,399 samples).
These MRI-adapted choices are registered before the full runs; do not tune on test.

This is an AuxK-loss ablation inspired by
[Gao et al., Appendix A.2](https://arxiv.org/html/2406.04093v1), not a reproduction
of their full training recipe. Untied initialization and decoder normalization
remain unchanged in BOTH arms to isolate the auxiliary loss. Their language-model
inactivity window would exceed this study's full training budget.

Interpret training inactivity separately from validation alive counts. Logs and
saved models include main-activation counts, inactivity ages, auxiliary batch use
and positive auxiliary activations. If the auxiliary branch is inactive or cannot
revive features, report that; it cannot establish robustness to all mitigations.
More alive units alone do not establish better features. Compare reconstruction,
eligible coverage, purity minus shuffle reference, and patient-distinct purity.
Raw-embedding probe results are identical between arms by design, not evidence
that SAE-code predictive utility is identical.

**Completed SAE weights save once per fit**, including transform, training activity,
history and protocol. No per-epoch checkpoints. A partly trained fit restarts;
completed fits and evaluation recovery reuse the final model. Export the full ZIP
before closing the cloud runtime. The export filename lists completed encoder seeds.

Live progress refreshes every 10 seconds. Stop requests finish the current fit.
This notebook evaluates previously inspected validation patients only. It does
not unlock held-out test data, calculate patient-bootstrap intervals or align
features across seeds. Those require separate evaluation after protocol freezing.
The accompanying held-out audit explains why older local test caches cannot be
substituted for the expanded-cohort encoders.


Local notebook: use the ncl_sae kernel. Run All starts the full registered plan. Edit SELECTED_SEEDS before running if splitting sessions.

In [ ]:
from pathlib import Path
import types,json,hashlib
RUNTIME_SOURCE='"""Expanded-cohort rank controls using archived CLS embeddings only."""\nfrom pathlib import Path\nimport gc,hashlib,itertools,json,os,random,sys,time,zipfile\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nDEFAULT_PLAN={\'version\':\'expanded-rank-control-v1\',\'methods\':[\'ncl\',\'simclr\'],\n \'encoder_seeds\':[10,42,52],\'sae_seeds\':[10,42,52],\'ranks\':[8],\n \'conditions\':[\'native\',\'pca_rank\',\'spectrum_matched\'],\'dictionary_size\':1536,\'k\':24,\n \'epochs\':30,\'batch_size\':512,\'lr\':0.0005,\'gradient_clip\':1.0,\'top_m\':10,\n \'permutations\':100,\'probe_C\':1.0,\'probe_max_iter\':2000,\n \'spectrum_reference\':\'NCL training covariance, same encoder seed\',\n \'save_completed_models\':True,\'per_epoch_checkpoints\':False,\n \'scope\':\'exploratory validation follow-up; no test images/features or spatial maps\'}\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(8*1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()\n\ndef replace_file(source,target):\n    # Windows readers may briefly deny replacement while the live UI reads CSVs.\n    for attempt in range(41):\n        try:\n            os.replace(source,target);return\n        except PermissionError:\n            if os.name!=\'nt\' or attempt==40:raise\n            time.sleep(.05)\n\ndef environment():\n    import sklearn\n    return {\'python\':sys.version,\'torch\':str(torch.__version__),\'numpy\':np.__version__,\n        \'pandas\':pd.__version__,\'scikit_learn\':sklearn.__version__,\'cuda\':torch.version.cuda,\n        \'gpu\':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}\n\ndef atomic_json(path,value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_name(path.name+\'.\'+str(os.getpid())+\'.tmp\')\n    tmp.write_text(json.dumps(value,indent=2,allow_nan=False),encoding=\'utf-8\');replace_file(tmp,path)\n\ndef atomic_csv(frame,path):\n    path=Path(path);tmp=path.with_name(path.name+\'.\'+str(os.getpid())+\'.tmp\')\n    frame.to_csv(tmp,index=False);replace_file(tmp,path)\n\ndef bundle_download(repo_id,filename,destination,expected_sha):\n    target=Path(destination).expanduser().resolve();target.parent.mkdir(parents=True,exist_ok=True)\n    if target.is_file():\n        if sha(target)!=expected_sha:raise ValueError(\'Input bundle SHA256 mismatch\')\n        return str(target)\n    token=os.environ.get(\'HF_TOKEN\',\'\').strip()\n    if not token:\n        from dotenv import dotenv_values\n        token=str(dotenv_values(\'.env\').get(\'HF_TOKEN\') or \'\').strip()\n    if not token:raise RuntimeError(\'Set read-only HF_TOKEN in runtime secrets or .env\')\n    from huggingface_hub import hf_hub_download\n    downloaded=Path(hf_hub_download(repo_id=repo_id,repo_type=\'dataset\',filename=filename,token=token,local_dir=str(target.parent))).resolve()\n    if sha(downloaded)!=expected_sha:raise ValueError(\'Input bundle SHA256 mismatch\')\n    if downloaded!=target:os.replace(downloaded,target)\n    return str(target)\n\nclass Inputs:\n    def __init__(self,bundle,cache,expected_sha):\n        self.bundle=Path(bundle).resolve();self.cache=Path(cache).resolve();self.cache.mkdir(parents=True,exist_ok=True)\n        signature={\'bytes\':self.bundle.stat().st_size,\'mtime_ns\':self.bundle.stat().st_mtime_ns,\'sha256\':expected_sha}\n        receipt=self.cache/\'bundle_verified.json\'\n        if not receipt.exists() or json.loads(receipt.read_text())!=signature:\n            if sha(self.bundle)!=expected_sha:raise ValueError(\'Wrong/corrupt embedding bundle\')\n            with zipfile.ZipFile(self.bundle) as z:\n                self.registration=json.loads(z.read(\'input_manifest.json\'))\n                if self.registration[\'format\']!=\'rank-control-embeddings-v1\':raise ValueError(\'Wrong bundle format\')\n                for name,identity in self.registration[\'files\'].items():\n                    target=(self.cache/name).resolve()\n                    if not target.is_relative_to(self.cache):raise ValueError(\'Unsafe archive path\')\n                    target.parent.mkdir(parents=True,exist_ok=True)\n                    temp=target.with_name(target.name+\'.tmp\');h=hashlib.sha256();count=0\n                    with z.open(name) as src,open(temp,\'wb\') as dst:\n                        while True:\n                            b=src.read(8*1024*1024)\n                            if not b:break\n                            h.update(b);count+=len(b);dst.write(b)\n                    if count!=identity[\'bytes\'] or h.hexdigest()!=identity[\'sha256\']:raise ValueError(\'Corrupt embedding file: \'+name)\n                    os.replace(temp,target)\n                atomic_json(self.cache/\'input_manifest.json\',self.registration)\n            atomic_json(receipt,signature)\n        self.registration=json.loads((self.cache/\'input_manifest.json\').read_text())\n        if sha(self.cache/\'split_manifest.csv\')!=self.registration[\'manifest_sha256\']:raise ValueError(\'Manifest changed\')\n        self.frame=pd.read_csv(self.cache/\'split_manifest.csv\',dtype={\'patient_id\':str,\'sample_id\':str})\n        if self.frame.duplicated([\'patient_id\',\'slice_idx\']).any() or self.frame.groupby(\'patient_id\').split.nunique().max()!=1:\n            raise ValueError(\'Duplicate samples or crossing patient partitions\')\n        self.pca_cache={}\n\n    def load(self,mode,seed,split):\n        if split not in (\'train\',\'val\'):raise ValueError(\'No test features available\')\n        prefix=self.cache/\'runs\'/mode/f\'encoder_{seed}\'\n        relative=f\'runs/{mode}/encoder_{seed}/{split}.npz\'\n        if sha(prefix/f\'{split}.npz\')!=self.registration[\'files\'][relative][\'sha256\']:raise ValueError(\'Cached input file changed\')\n        meta=json.loads((prefix/f\'{split}_identity.json\').read_text())\n        expected=self.frame.loc[self.frame.split==split,\'sample_id\'].astype(str).to_numpy()\n        if meta[\'manifest\']!=self.registration[\'manifest_sha256\'] or meta[\'samples\']!=hashlib.sha256(\'\\n\'.join(expected).encode()).hexdigest():\n            raise ValueError(\'Embedding identity mismatch\')\n        with np.load(prefix/f\'{split}.npz\',allow_pickle=False) as data:\n            x=data[\'x\'].astype(np.float32);ids=data[\'sample_ids\']\n        if not np.array_equal(ids,expected) or x.shape!=(len(expected),192) or not np.isfinite(x).all():raise ValueError(\'Invalid embedding cache\')\n        return x\n\n    def pca(self,mode,seed,x=None):\n        key=(mode,seed)\n        if key not in self.pca_cache:\n            x=self.load(mode,seed,\'train\') if x is None else x\n            mean=x.astype(np.float64).mean(0);cov=covariance(x)\n            eig,basis=np.linalg.eigh(cov);order=np.argsort(eig)[::-1]\n            self.pca_cache[key]={\'mean\':mean.astype(np.float32),\'eigenvalues\':np.maximum(eig[order],0),\'basis\':basis[:,order]}\n        return self.pca_cache[key]\n\ndef covariance(x):\n    mean=np.mean(x,axis=0,dtype=np.float64);cov=np.zeros((x.shape[1],x.shape[1]),dtype=np.float64)\n    for start in range(0,len(x),8192):\n        centered=x[start:start+8192].astype(np.float64)-mean;cov+=centered.T@centered\n    return cov/max(1,len(x)-1)\n\ndef spectrum_stats(eigenvalues):\n    v=np.maximum(eigenvalues,0);total=float(v.sum());p=v/total if total else v\n    p=p[p>0]\n    return {\'variance\':total,\'effective_rank\':float(np.exp(-np.sum(p*np.log(p)))) if total else 0,\n            \'participation_ratio\':float(total**2/np.sum(v*v)) if total else 0}\n\ndef build_transform(inputs,mode,seed,condition,rank,train,val):\n    info=inputs.pca(mode,seed,train);mean=info[\'mean\'];centered_train=train-mean;centered_val=val-mean\n    if condition==\'native\':\n        basis=np.eye(192);scale=np.ones(192);rank=192\n        ztrain=centered_train;zval=centered_val;target=None;pca_ev=1.0\n    else:\n        basis=info[\'basis\'][:,:rank];values=info[\'eigenvalues\'][:rank]\n        if np.min(values)<=info[\'eigenvalues\'][0]*1e-12:raise ValueError(\'Requested rank contains numerically singular dimensions\')\n        scale=np.ones(rank)\n        target=None\n        if condition==\'spectrum_matched\':\n            target=inputs.pca(\'ncl\',seed)[\'eigenvalues\'][:rank]\n            if np.min(target)<=0:raise ValueError(\'NCL reference covariance is singular\')\n            scale=np.sqrt(target/values)\n        elif condition!=\'pca_rank\':raise ValueError(\'Unknown control condition\')\n        ztrain=((centered_train@basis)*scale)@basis.T;zval=((centered_val@basis)*scale)@basis.T\n        unscaled=(centered_val@basis)@basis.T\n        pca_ev=1-float(np.sum((centered_val.astype(np.float64)-unscaled)**2))/float(np.sum(centered_val.astype(np.float64)**2))\n    ztrain=np.asarray(ztrain,dtype=np.float32);zval=np.asarray(zval,dtype=np.float32)\n    train_eig=np.maximum(np.linalg.eigvalsh(covariance(ztrain)),0)[::-1]\n    val_eig=np.maximum(np.linalg.eigvalsh(covariance(zval)),0)[::-1]\n    diagnostics={\'condition\':condition,\'rank\':rank,\'PCA_only_validation_EV\':pca_ev,\n        \'PCA_train_variance_retained\':float(info[\'eigenvalues\'][:rank].sum()/info[\'eigenvalues\'].sum()),\n        \'train\':spectrum_stats(train_eig),\'validation\':spectrum_stats(val_eig),\n        \'matching_scope\':\'train only; validation spectra are measured, never forced\'}\n    if target is not None:\n        error=float(np.linalg.norm(train_eig[:rank]-target)/np.linalg.norm(target))\n        if error>1e-4:raise ValueError(\'Training spectrum matching failed\')\n        diagnostics.update(target=spectrum_stats(target),relative_training_spectrum_error=error)\n    recipe={\'mean\':mean,\'basis\':basis.astype(np.float32),\'scale\':scale.astype(np.float32),\'condition\':condition,\'rank\':rank}\n    return ztrain,zval,recipe,diagnostics\n\nclass SAE(nn.Module):\n    def __init__(self,width,dictionary,k):\n        super().__init__();self.k=k;self.pre_bias=nn.Parameter(torch.zeros(width))\n        self.encoder=nn.Linear(width,dictionary);self.decoder=nn.Linear(dictionary,width,bias=False)\n    def forward(self,x):\n        raw=F.relu(self.encoder(x-self.pre_bias));v,i=raw.topk(self.k,dim=-1)\n        acts=torch.zeros_like(raw).scatter_(-1,i,v)\n        return self.decoder(acts)+self.pre_bias,acts\n\ndef seed_all(seed):\n    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False\n\ndef purity_scores(top,labels,classes):\n    if len(top)==0:return np.empty(0),np.empty(0)\n    counts=np.stack([(labels[top]==label).sum(axis=1) for label in classes],axis=1)\n    p=counts/top.shape[1];entropy=-np.sum(np.where(p>0,p*np.log(np.maximum(p,1e-300)),0),axis=1)\n    return p.max(axis=1),entropy\n\ndef selectivity(acts,labels,patients,m,permutations,seed):\n    support=(acts>0).sum(0);alive=np.flatnonzero(support);eligible=np.flatnonzero(support>=m)\n    top=np.argsort(acts[:,eligible],axis=0)[-m:].T if len(eligible) else np.empty((0,m),int)\n    classes=np.unique(labels);purity,entropy=purity_scores(top,labels,classes)\n    patient_top=[];patient_features=[]\n    for feature in eligible:\n        chosen=[];seen=set()\n        for index in np.argsort(acts[:,feature])[::-1]:\n            if acts[index,feature]<=0:break\n            if patients[index] in seen:continue\n            seen.add(patients[index]);chosen.append(index)\n            if len(chosen)==m:break\n        if len(chosen)==m:patient_features.append(feature);patient_top.append(chosen)\n    patient_top=np.asarray(patient_top,dtype=int).reshape(-1,m)\n    pp,pe=purity_scores(patient_top,labels,classes)\n    rng=np.random.default_rng(seed);null=[];pnull=[]\n    for _ in range(permutations):\n        shuffled=rng.permutation(labels)\n        a,_=purity_scores(top,shuffled,classes);b,_=purity_scores(patient_top,shuffled,classes)\n        if len(a):null.append(float(a.mean()))\n        if len(b):pnull.append(float(b.mean()))\n    def average(v):return float(np.mean(v)) if len(v) else None\n    result={\'alive_features\':len(alive),\'eligible_features\':len(eligible),\'mean_purity\':average(purity),\n        \'mean_entropy_nats\':average(entropy),\'permutation_mean_purity\':average(null),\n        \'patient_distinct_eligible_features\':len(patient_features),\'patient_distinct_purity\':average(pp),\n        \'patient_distinct_permutation_mean_purity\':average(pnull),\'patient_distinct_entropy\':average(pe),\n        \'mean_positive_features_per_slice\':float((acts>0).sum(1).mean()),\n        \'permutation_scope\':\'descriptive slice-label shuffle; not a patient-level significance test\'}\n    result[\'purity_minus_permutation\']=result[\'mean_purity\']-result[\'permutation_mean_purity\'] if null else None\n    result[\'patient_distinct_purity_minus_permutation\']=result[\'patient_distinct_purity\']-result[\'patient_distinct_permutation_mean_purity\'] if pnull else None\n    table=pd.DataFrame({\'feature\':eligible,\'positive_support\':support[eligible],\'purity\':purity,\'entropy_nats\':entropy})\n    patient_table=pd.DataFrame({\'feature\':patient_features,\'purity\':pp,\'entropy_nats\':pe})\n    return result,table,patient_table\n\ndef evaluate(model,zval,original_val,recipe,labels,patients,plan,sae_seed):\n    device=next(model.parameters()).device;codes=[];sse=0.;original_sse=0.\n    mean=recipe[\'mean\'];basis=recipe[\'basis\'];scale=recipe[\'scale\']\n    centered=original_val.astype(np.float64)-mean\n    with torch.no_grad():\n        for start in range(0,len(zval),512):\n            x=zval[start:start+512];recon,acts=model(torch.from_numpy(x).to(device))\n            r=recon.cpu().numpy();codes.append(acts.cpu().numpy())\n            sse+=float(np.sum((x.astype(np.float64)-r)**2))\n            restored=r if recipe[\'condition\']==\'native\' else ((r@basis)/scale)@basis.T\n            original_sse+=float(np.sum((centered[start:start+len(x)]-restored)**2))\n    acts=np.concatenate(codes);reference=float(np.sum(zval.astype(np.float64)**2))\n    metrics={\'normalized_SAE_EV\':1-sse/reference,\'original_coordinate_EV\':1-original_sse/float(np.sum(centered**2)),\'mse\':sse/zval.size}\n    concept,features,patient_features=selectivity(acts,labels,patients,plan[\'top_m\'],plan[\'permutations\'],sae_seed)\n    metrics.update(concept)\n    return metrics,features,patient_features\n\ndef fit_one(ztrain,zval,original_val,recipe,diagnostics,labels,patients,plan,mode,encoder_seed,sae_seed,folder,identity):\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\n    protocol={**identity,\'method\':mode,\'encoder_seed\':encoder_seed,\'sae_seed\':sae_seed,\n        \'condition\':recipe[\'condition\'],\'rank\':recipe[\'rank\'],\'plan\':plan}\n    registration=folder/\'protocol.json\'\n    if registration.exists() and json.loads(registration.read_text())!=protocol:raise ValueError(\'Fit protocol changed; choose new output\')\n    atomic_json(registration,protocol)\n    complete=folder/\'validation_metrics.json\';model_path=folder/\'completed_model.pt\'\n    if complete.exists():\n        r=json.loads(complete.read_text())\n        if not model_path.exists() or sha(model_path)!=r[\'model_sha256\']:raise ValueError(\'Missing/corrupt completed dictionary\')\n        print(\'Reusing completed control:\',mode,encoder_seed,sae_seed,recipe[\'condition\'],recipe[\'rank\'],flush=True)\n        return r\n    seed_all(sae_seed);device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    model=SAE(ztrain.shape[1],plan[\'dictionary_size\'],plan[\'k\']).to(device)\n    if model_path.exists():\n        saved=torch.load(model_path,map_location=\'cpu\',weights_only=True)\n        if saved[\'protocol\']!=protocol:raise ValueError(\'Completed model protocol mismatch\')\n        model.load_state_dict(saved[\'model\']);history=saved[\'history\']\n    else:\n        optimizer=torch.optim.Adam(model.parameters(),lr=plan[\'lr\']);history=[]\n        loader=torch.utils.data.DataLoader(torch.utils.data.TensorDataset(torch.from_numpy(ztrain)),batch_size=plan[\'batch_size\'],shuffle=True)\n        for epoch in range(plan[\'epochs\']):\n            model.train();total=count=0\n            for (batch,) in loader:\n                batch=batch.to(device);optimizer.zero_grad(set_to_none=True);recon,_=model(batch)\n                loss=F.mse_loss(recon,batch)\n                if not torch.isfinite(loss):raise ValueError(\'Non-finite loss\')\n                loss.backward();nn.utils.clip_grad_norm_(model.parameters(),plan[\'gradient_clip\']);optimizer.step()\n                total+=float(loss.detach())*len(batch);count+=len(batch)\n            history.append({\'epoch\':epoch+1,\'train_mse\':total/count})\n            atomic_json(folder/\'history.json\',history)\n            print(mode,encoder_seed,recipe[\'condition\'],recipe[\'rank\'],\'SAE\',sae_seed,\'epoch\',epoch+1,\'mse\',total/count,flush=True)\n        # One final artifact, no optimizer/scaler/RNG or per-epoch model writes.\n        payload={\'model\':{k:v.detach().cpu() for k,v in model.state_dict().items()},\'protocol\':protocol,\'history\':history,\'training_environment\':environment(),\n            \'transform\':{k:torch.from_numpy(v) if isinstance(v,np.ndarray) else v for k,v in recipe.items()}}\n        temp=model_path.with_suffix(\'.tmp\');torch.save(payload,temp);replace_file(temp,model_path)\n    model.eval();metrics,features,patient_features=evaluate(model,zval,original_val,recipe,labels,patients,plan,sae_seed)\n    features.to_csv(folder/\'validation_features.csv\',index=False);patient_features.to_csv(folder/\'patient_distinct_features.csv\',index=False)\n    result={**metrics,\'method\':mode,\'encoder_seed\':encoder_seed,\'sae_seed\':sae_seed,\'condition\':recipe[\'condition\'],\n        \'rank\':recipe[\'rank\'],\'k\':plan[\'k\'],\'epochs\':plan[\'epochs\'],\'split\':\'val\',\'diagnostics\':diagnostics,\n        \'model_sha256\':sha(model_path),\'selection_scope\':\'exploratory controls; no test evaluation\'}\n    atomic_json(complete,result);del model;gc.collect()\n    if torch.cuda.is_available():torch.cuda.empty_cache()\n    return result\n\ndef probe(ztrain,zval,labels_train,labels_val,plan):\n    from sklearn.pipeline import make_pipeline\n    from sklearn.preprocessing import StandardScaler\n    from sklearn.linear_model import LogisticRegression\n    model=make_pipeline(StandardScaler(),LogisticRegression(C=plan[\'probe_C\'],max_iter=plan[\'probe_max_iter\']))\n    model.fit(ztrain,labels_train);predictions=model.predict(zval)\n    return {\'slice_accuracy\':float(np.mean(predictions==labels_val)),\'train_fit_only\':True,\'C\':plan[\'probe_C\'],\'max_iter\':plan[\'probe_max_iter\']}\n\ndef run(bundle,bundle_sha,output,plan=None,seeds=None,methods=None,max_hours=8,require_cuda=True):\n    plan=json.loads(json.dumps(DEFAULT_PLAN if plan is None else plan));seeds=plan[\'encoder_seeds\'] if seeds is None else list(seeds)\n    methods=plan[\'methods\'] if methods is None else list(methods)\n    if not seeds or set(seeds)-set(plan[\'encoder_seeds\']) or not methods or set(methods)-set(plan[\'methods\']):raise ValueError(\'Invalid method/seed selection\')\n    if require_cuda and not torch.cuda.is_available():raise RuntimeError(\'Attach a CUDA GPU\')\n    root=Path(output).expanduser().resolve();root.mkdir(parents=True,exist_ok=True)\n    started=time.monotonic();inputs=Inputs(bundle,root/\'input_cache\',bundle_sha)\n    identity={\'bundle_sha256\':bundle_sha,\'manifest_sha256\':inputs.registration[\'manifest_sha256\'],\n        \'runtime_sha256\':globals().get(\'RUNTIME_SHA256\',\'development\'),\'plan\':plan}\n    registration=root/\'protocol.json\'\n    if registration.exists() and json.loads(registration.read_text())!=identity:raise ValueError(\'Control protocol/code changed; choose a new output folder\')\n    atomic_json(registration,identity)\n    atomic_json(root/\'environment.json\',environment())\n    val=inputs.frame[inputs.frame.split==\'val\'];train_frame=inputs.frame[inputs.frame.split==\'train\']\n    labels=val.dominant_label.to_numpy();patients=val.patient_id.to_numpy()\n    status=\'completed selected controls\'\n    for seed,mode in itertools.product(seeds,methods):\n        train=inputs.load(mode,seed,\'train\');validation=inputs.load(mode,seed,\'val\')\n        cases=[(\'native\',192)]+[(condition,rank) for condition in plan[\'conditions\'] if condition!=\'native\' for rank in plan[\'ranks\']]\n        for condition,rank in cases:\n            if (root/\'stop_after_fit.flag\').exists() or time.monotonic()-started>=max_hours*3600:\n                status=\'stopped between fits; completed dictionaries retained\';break\n            ztrain,zval,recipe,diagnostics=build_transform(inputs,mode,seed,condition,rank,train,validation)\n            case_dir=root/\'fits\'/mode/f\'encoder_{seed}\'/f\'{condition}_r{rank}\'\n            case_dir.mkdir(parents=True,exist_ok=True);atomic_json(case_dir/\'transform_diagnostics.json\',diagnostics)\n            probe_path=case_dir/\'validation_probe.json\'\n            if not probe_path.exists():atomic_json(probe_path,probe(ztrain,zval,train_frame.has_tumor.to_numpy(),val.has_tumor.to_numpy(),plan))\n            for sae_seed in plan[\'sae_seeds\']:\n                if (root/\'stop_after_fit.flag\').exists() or time.monotonic()-started>=max_hours*3600:\n                    status=\'stopped between fits; completed dictionaries retained\';break\n                fit_one(ztrain,zval,validation,recipe,diagnostics,labels,patients,plan,mode,seed,sae_seed,case_dir/f\'sae_{sae_seed}\',identity)\n                summarize(root)\n            del ztrain,zval;gc.collect()\n            if status!=\'completed selected controls\':break\n        del train,validation;gc.collect()\n        if status!=\'completed selected controls\':break\n    report={\'status\':status,\'encoder_seeds\':seeds,\'methods\':methods,\'elapsed_hours\':(time.monotonic()-started)/3600,\n        \'test_evaluated\':False,\'encoders_retrained\':False,\'per_epoch_checkpoints\':False,\'completed_models_saved_once\':True}\n    atomic_json(root/\'last_session.json\',report);summarize(root)\n    return report\n\ndef summarize(root):\n    root=Path(root);records=[json.loads(p.read_text()) for p in root.glob(\'fits/*/encoder_*/*/sae_*/validation_metrics.json\')]\n    if not records:return pd.DataFrame(),pd.DataFrame(),pd.DataFrame()\n    frame=pd.json_normalize(records);group=[\'method\',\'encoder_seed\',\'condition\',\'rank\']\n    metrics=[\'normalized_SAE_EV\',\'original_coordinate_EV\',\'alive_features\',\'eligible_features\',\'mean_purity\',\'purity_minus_permutation\',\n        \'patient_distinct_purity\',\'patient_distinct_purity_minus_permutation\',\'patient_distinct_eligible_features\']\n    means=frame.groupby(group)[metrics].mean().reset_index()\n    means[\'sae_replicates\']=frame.groupby(group).size().to_numpy()\n    # Only complete sets of SAE seeds enter the across-encoder summary.\n    plan=json.loads((root/\'protocol.json\').read_text())[\'plan\'];complete=means[means.sae_replicates==len(plan[\'sae_seeds\'])]\n    summary=complete.groupby([\'method\',\'condition\',\'rank\'])[metrics].agg([\'mean\',\'std\',\'count\']).reset_index()\n    summary.columns=[\'_\'.join(str(v) for v in c if str(v)) if isinstance(c,tuple) else c for c in summary.columns]\n    atomic_csv(frame,root/\'all_control_fits.csv\');atomic_csv(means,root/\'per_encoder_controls.csv\');atomic_csv(summary,root/\'control_summary.csv\')\n    differences=[]\n    for (seed,condition,rank),g in complete.groupby([\'encoder_seed\',\'condition\',\'rank\']):\n        if set(g.method)!={\'ncl\',\'simclr\'}:continue\n        a=g[g.method==\'ncl\'].iloc[0];b=g[g.method==\'simclr\'].iloc[0]\n        differences.append({\'encoder_seed\':int(seed),\'condition\':condition,\'rank\':int(rank),\n            **{\'ncl_minus_simclr_\'+m:float(a[m]-b[m]) for m in metrics}})\n    atomic_csv(pd.DataFrame(differences),root/\'paired_control_differences.csv\')\n    return frame,means,summary\n\ndef export_results(root):\n    root=Path(root).resolve()\n    seeds=sorted({json.loads(p.read_text())[\'encoder_seed\'] for p in root.glob(\'fits/*/encoder_*/*/sae_*/validation_metrics.json\')})\n    suffix=\'_enc\'+\'-\'.join(str(s) for s in seeds) if seeds else \'\'\n    target=root.parent/(root.name+suffix+\'_ALL_RESULTS.zip\')\n    with zipfile.ZipFile(target,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(root.rglob(\'*\')):\n            if not p.is_file() or \'input_cache\' in p.relative_to(root).parts or p.name in (\'.env\',\'worker_request.json\'):continue\n            if p.suffix in (\'.json\',\'.csv\',\'.pt\',\'.log\',\'.txt\',\'.png\'):z.write(p,p.relative_to(root))\n    return str(target)\n\n"""Appended to the frozen rank runtime; overrides only the SAE experiment loop."""\nDEFAULT_PLAN = {**DEFAULT_PLAN, \'version\':\'expanded-auxk-control-v1\',\n \'conditions\':[\'baseline\',\'auxk\'], \'ranks\':[], \'auxk_coefficient\':1/32,\n \'auxk_k\':192, \'dead_after_training_passes\':1.0,\n \'auxk_target\':\'detached main reconstruction residual; no decoder bias\',\n \'initialization\':\'unchanged torch-default untied initialization in both arms\',\n \'decoder_normalization\':False,\n \'scope\':\'native centered archived CLS; exploratory validation; not full Gao recipe\'}\n\ndef auxk_objective(model,batch,ages,threshold,aux_k):\n    """Top-K positive dead-latent residual reconstruction, sharing the encoder."""\n    raw=F.relu(model.encoder(batch-model.pre_bias))\n    values,indices=raw.topk(model.k,dim=-1)\n    acts=torch.zeros_like(raw).scatter_(-1,indices,values)\n    recon=model.decoder(acts)+model.pre_bias\n    # Main activity only resets dead status. Auxiliary activations never reset it.\n    rows=torch.arange(len(batch),device=batch.device,dtype=torch.long)[:,None]+1\n    last=torch.where(acts>0,rows,0).amax(0)\n    updated=torch.where(last>0,len(batch)-last,ages+len(batch))\n    dead=updated>=threshold\n    n_dead=int(dead.sum().item())\n    aux=recon.sum()*0\n    positive_aux=0\n    if n_dead:\n        masked=raw.masked_fill(~dead[None,:],float(\'-inf\'))\n        v,i=masked.topk(min(aux_k,n_dead),dim=-1)\n        code=torch.zeros_like(raw).scatter_(-1,i,v)\n        residual=(batch-recon).detach()\n        aux=F.mse_loss(model.decoder(code),residual)\n        positive_aux=int((code>0).sum().item())\n    return recon,acts,aux,updated,n_dead,positive_aux\n\ndef fit_one(ztrain,zval,original_val,recipe,diagnostics,labels,patients,plan,mode,encoder_seed,sae_seed,folder,identity):\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\n    condition=recipe[\'condition\']\n    protocol={**identity,\'method\':mode,\'encoder_seed\':encoder_seed,\'sae_seed\':sae_seed,\n        \'condition\':condition,\'rank\':192,\'plan\':plan}\n    registration=folder/\'protocol.json\'\n    if registration.exists() and json.loads(registration.read_text())!=protocol:\n        raise ValueError(\'Fit protocol changed; choose a fresh output folder\')\n    atomic_json(registration,protocol)\n    complete=folder/\'validation_metrics.json\';model_path=folder/\'completed_model.pt\'\n    if complete.exists():\n        record=json.loads(complete.read_text())\n        if not model_path.exists() or sha(model_path)!=record[\'model_sha256\']:raise ValueError(\'Missing/corrupt completed model\')\n        return record\n    seed_all(sae_seed);device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    model=SAE(192,plan[\'dictionary_size\'],plan[\'k\']).to(device)\n    threshold=max(1,int(np.ceil(len(ztrain)*plan[\'dead_after_training_passes\'])))\n    if model_path.exists():\n        saved=torch.load(model_path,map_location=\'cpu\',weights_only=True)\n        if saved[\'protocol\']!=protocol:raise ValueError(\'Saved model protocol mismatch\')\n        model.load_state_dict(saved[\'model\']);history=saved[\'history\'];train_activity=saved[\'training_activity\']\n    else:\n        optimizer=torch.optim.Adam(model.parameters(),lr=plan[\'lr\'])\n        loader=torch.utils.data.DataLoader(torch.utils.data.TensorDataset(torch.from_numpy(ztrain)),\n            batch_size=plan[\'batch_size\'],shuffle=True)\n        ages=torch.zeros(plan[\'dictionary_size\'],dtype=torch.long,device=device)\n        counts=torch.zeros_like(ages);history=[]\n        for epoch in range(plan[\'epochs\']):\n            total=aux_total=objective_total=0.;seen=aux_batches=aux_positive=0;max_dead=0\n            model.train()\n            for (batch,) in loader:\n                batch=batch.to(device);optimizer.zero_grad(set_to_none=True)\n                if condition==\'auxk\':\n                    recon,acts,aux,ages,n_dead,n_positive=auxk_objective(model,batch,ages,threshold,plan[\'auxk_k\'])\n                    aux_batches+=int(n_dead>0);aux_positive+=n_positive;max_dead=max(max_dead,n_dead)\n                else:\n                    recon,acts=model(batch);aux=recon.sum()*0\n                    rows=torch.arange(len(batch),device=device,dtype=torch.long)[:,None]+1\n                    last=torch.where(acts>0,rows,0).amax(0)\n                    ages=torch.where(last>0,len(batch)-last,ages+len(batch))\n                counts+=(acts.detach()>0).sum(0)\n                main=F.mse_loss(recon,batch)\n                objective=main+plan[\'auxk_coefficient\']*aux\n                if not torch.isfinite(objective):raise ValueError(\'Non-finite AuxK objective\')\n                objective.backward();nn.utils.clip_grad_norm_(model.parameters(),plan[\'gradient_clip\']);optimizer.step()\n                total+=float(main.detach())*len(batch);aux_total+=float(aux.detach())*len(batch)\n                objective_total+=float(objective.detach())*len(batch);seen+=len(batch)\n            item={\'epoch\':epoch+1,\'train_mse\':total/seen,\'auxiliary_mse\':aux_total/seen,\n                \'objective\':objective_total/seen,\'auxiliary_batches\':aux_batches,\n                \'positive_auxiliary_activations\':aux_positive,\'max_dead_candidates\':max_dead,\n                \'dead_at_epoch_end\':int((ages>=threshold).sum()),\'alive_training_cumulative\':int((counts>0).sum())}\n            history.append(item);atomic_json(folder/\'history.json\',history)\n            print(mode,encoder_seed,condition,\'SAE\',sae_seed,item,flush=True)\n        train_activity={\'positive_counts\':counts.cpu(),\'samples_since_main_activation\':ages.cpu(),\n            \'dead_threshold_samples\':threshold,\'auxiliary_batches\':sum(h[\'auxiliary_batches\'] for h in history),\n            \'positive_auxiliary_activations\':sum(h[\'positive_auxiliary_activations\'] for h in history)}\n        payload={\'model\':{k:v.detach().cpu() for k,v in model.state_dict().items()},\'protocol\':protocol,\n            \'history\':history,\'training_activity\':train_activity,\'training_environment\':environment(),\n            \'transform\':{k:torch.from_numpy(v) if isinstance(v,np.ndarray) else v for k,v in recipe.items()}}\n        temp=model_path.with_suffix(\'.tmp\');torch.save(payload,temp);replace_file(temp,model_path)\n    model.eval()\n    metrics,features,patient_features=evaluate(model,zval,original_val,recipe,labels,patients,plan,sae_seed)\n    atomic_csv(features,folder/\'validation_features.csv\');atomic_csv(patient_features,folder/\'patient_distinct_features.csv\')\n    counts=train_activity[\'positive_counts\'];ages=train_activity[\'samples_since_main_activation\']\n    metrics.update(training_alive_features=int((counts>0).sum()),training_dead_at_end=int((ages>=threshold).sum()),\n        auxiliary_batches=train_activity[\'auxiliary_batches\'],positive_auxiliary_activations=train_activity[\'positive_auxiliary_activations\'])\n    result={**metrics,\'method\':mode,\'encoder_seed\':encoder_seed,\'sae_seed\':sae_seed,\'condition\':condition,\n        \'rank\':192,\'k\':plan[\'k\'],\'epochs\':plan[\'epochs\'],\'split\':\'val\',\'diagnostics\':diagnostics,\n        \'model_sha256\':sha(model_path),\'selection_scope\':\'exploratory AuxK control; no test evaluation\'}\n    atomic_json(complete,result);del model;gc.collect()\n    if torch.cuda.is_available():torch.cuda.empty_cache()\n    return result\n\n_base_summarize=summarize\ndef summarize(root):\n    frame,means,summary=_base_summarize(root)\n    differences=[]\n    if len(means):\n        for (mode,seed),g in means.groupby([\'method\',\'encoder_seed\']):\n            expected=len(json.loads((Path(root)/\'protocol.json\').read_text())[\'plan\'][\'sae_seeds\'])\n            g=g[g.sae_replicates==expected]\n            if set(g.condition)!={\'baseline\',\'auxk\'}:continue\n            a=g[g.condition==\'auxk\'].iloc[0];b=g[g.condition==\'baseline\'].iloc[0]\n            metrics=[\'alive_features\',\'eligible_features\',\'normalized_SAE_EV\',\'mean_purity\',\'purity_minus_permutation\',\n                \'patient_distinct_purity\',\'patient_distinct_purity_minus_permutation\']\n            differences.append({\'method\':mode,\'encoder_seed\':int(seed),\n                **{\'auxk_minus_baseline_\'+m:float(a[m]-b[m]) for m in metrics}})\n    atomic_csv(pd.DataFrame(differences),Path(root)/\'auxk_minus_baseline.csv\')\n    return frame,means,summary\n\ndef run(bundle,bundle_sha,output,plan=None,seeds=None,methods=None,max_hours=8,require_cuda=True):\n    plan=json.loads(json.dumps(DEFAULT_PLAN if plan is None else plan))\n    seeds=plan[\'encoder_seeds\'] if seeds is None else list(seeds)\n    methods=plan[\'methods\'] if methods is None else list(methods)\n    if not seeds or set(seeds)-set(plan[\'encoder_seeds\']) or not methods or set(methods)-set(plan[\'methods\']):raise ValueError(\'Invalid method/seed selection\')\n    if plan[\'conditions\']!=[\'baseline\',\'auxk\'] or plan[\'auxk_k\']<1 or plan[\'auxk_coefficient\']<=0 or plan[\'dead_after_training_passes\']<=0:\n        raise ValueError(\'Invalid registered AuxK plan\')\n    if require_cuda and not torch.cuda.is_available():raise RuntimeError(\'Attach a CUDA GPU\')\n    root=Path(output).expanduser().resolve();root.mkdir(parents=True,exist_ok=True)\n    started=time.monotonic();inputs=Inputs(bundle,root/\'input_cache\',bundle_sha)\n    identity={\'bundle_sha256\':bundle_sha,\'manifest_sha256\':inputs.registration[\'manifest_sha256\'],\n        \'runtime_sha256\':globals().get(\'RUNTIME_SHA256\',\'development\'),\'plan\':plan}\n    registration=root/\'protocol.json\'\n    if registration.exists() and json.loads(registration.read_text())!=identity:raise ValueError(\'Protocol changed; use a fresh AuxK output folder\')\n    atomic_json(registration,identity);atomic_json(root/\'environment.json\',environment())\n    val=inputs.frame[inputs.frame.split==\'val\'];train_frame=inputs.frame[inputs.frame.split==\'train\']\n    status=\'completed selected controls\'\n    for seed,mode in itertools.product(seeds,methods):\n        train=inputs.load(mode,seed,\'train\');validation=inputs.load(mode,seed,\'val\')\n        ztrain,zval,base_recipe,diagnostics=build_transform(inputs,mode,seed,\'native\',192,train,validation)\n        probe_result=None\n        for condition in plan[\'conditions\']:\n            recipe={**base_recipe,\'condition\':condition};case_dir=root/\'fits\'/mode/f\'encoder_{seed}\'/f\'{condition}_r192\'\n            case_dir.mkdir(parents=True,exist_ok=True)\n            atomic_json(case_dir/\'transform_diagnostics.json\',diagnostics)\n            probe_path=case_dir/\'validation_probe.json\'\n            if not probe_path.exists():\n                if probe_result is None:probe_result=probe(ztrain,zval,train_frame.has_tumor.to_numpy(),val.has_tumor.to_numpy(),plan)\n                atomic_json(probe_path,probe_result)\n            for sae_seed in plan[\'sae_seeds\']:\n                if (root/\'stop_after_fit.flag\').exists() or time.monotonic()-started>=max_hours*3600:\n                    status=\'stopped between fits; completed dictionaries retained\';break\n                fit_one(ztrain,zval,validation,recipe,diagnostics,val.dominant_label.to_numpy(),val.patient_id.to_numpy(),\n                    plan,mode,seed,sae_seed,case_dir/f\'sae_{sae_seed}\',identity)\n                summarize(root)\n            if status!=\'completed selected controls\':break\n        del train,validation,ztrain,zval;gc.collect()\n        if status!=\'completed selected controls\':break\n    report={\'status\':status,\'encoder_seeds\':seeds,\'methods\':methods,\'elapsed_hours\':(time.monotonic()-started)/3600,\n        \'test_evaluated\':False,\'encoders_retrained\':False,\'per_epoch_checkpoints\':False,\'completed_models_saved_once\':True}\n    atomic_json(root/\'last_session.json\',report);summarize(root);return report\n'
engine=types.ModuleType('rank_control_runtime')
exec(compile(RUNTIME_SOURCE,'rank_control_runtime','exec'),engine.__dict__)
engine.RUNTIME_SHA256=hashlib.sha256(RUNTIME_SOURCE.encode()).hexdigest()
BUNDLE_SHA256='e41d4ae24932127a5747073d7225bdfd2d20022ca8a22027222674b7494cf9eb'


In [ ]:
start=Path.cwd().resolve()
ROOT=next((p for p in [start,*start.parents] if (p/'data/rank_control_embeddings_v1.zip').is_file()),start)
BUNDLE=ROOT/'data/rank_control_embeddings_v1.zip'
OUT=ROOT/'results/auxk_native_controls_v1'
PLAN=json.loads(json.dumps(engine.DEFAULT_PLAN))
SELECTED_SEEDS=[10,42,52]
METHODS=['ncl','simclr']
print('SAE fits:',len(SELECTED_SEEDS)*len(METHODS)*len(PLAN['sae_seeds'])*len(PLAN['conditions']))
print('No encoder training; no raw MRI input; completed SAE models save once.')


In [ ]:
report=engine.run(BUNDLE,BUNDLE_SHA256,OUT,PLAN,seeds=SELECTED_SEEDS,methods=METHODS,max_hours=8)
print(report)


In [ ]:
from IPython.display import display
fits,per_encoder,summary=engine.summarize(OUT)
display(per_encoder)
display(summary)
print('Mean/SD across encoders are meaningful only after all three encoder seeds finish.')


In [ ]:
archive=engine.export_results(OUT)
print('Full results and completed models:',archive)
